# See Water through the Years and Seasons

Explore rainfall across years and seasons, then compare fortnightly water and vegetation.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
pd.set_option("display.max_colwidth", 160)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the location

These three fields contain the selected tehsil when downloaded from GeoLibre. Edit them to explore another location, then restart the kernel and run from the top.


In [ ]:
state = "Bihar"
district = "Nalanda"
tehsil = "Hilsa"


## Set your API key

The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and keys. This cell reuses `CORE_STACK_API_KEY` or asks privately and stores it in this kernel’s environment. The request header is `X-API-Key`.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", value.replace("(", "").replace(")", "")).strip("_").lower()
         for key, value in {"state": state, "district": district, "tehsil": tehsil}.items()}
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}


## Read one table and choose a record

The tehsil API has no table or column filter. This cell reads it once and selects a few fields. For other tables, use `pd.DataFrame(api_data[table_name])` and select `columns` from that table; the response is already in memory. The field list shows the available names.


In [ ]:
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
display(pd.DataFrame({"table": list(api_data), "rows": [len(rows) for rows in api_data.values()]}))
table_name = 'hydrological_annual'
table = pd.DataFrame(api_data[table_name])

display(pd.DataFrame({"field": table.columns}))

examples = table

display(examples[['uid']].head(10))
mws_id = str(examples.iloc[0]["uid"])  # Replace with an identifier from the table.
selected = table.loc[table["uid"].astype(str) == mws_id].iloc[0]
columns = ['uid', 'precipitation_in_mm_2017-2018', 'et_in_mm_2017-2018', 'runoff_in_mm_2017-2018']
display(selected.reindex(columns).to_frame("value"))


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, "{state}/{district}/{tehsil}/collection.json".format(**place))
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "water_balance_fortnightly_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
field_notes = pd.DataFrame(columns=["name", "type", "description"])
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(pd.DataFrame([item["properties"]]).reindex(columns=["title", "description", "start_datetime", "end_datetime"]).T)
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Annual and seasonal rainfall

Change `measure` to `et` or `runoff` for the other water measures. The source field names stay in the table. Each chart uses millimetres and the same year positions; the annual and seasonal summaries are published separately.


In [ ]:
measure = "precipitation"
annual = pd.DataFrame(api_data["hydrological_annual"]).set_index("uid").reindex([mws_id]).iloc[0]
seasonal = pd.DataFrame(api_data["hydrological_seasonal"]).set_index("uid").reindex([mws_id]).iloc[0]
fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True, sharey=True)
values = []
for ax, period in zip(axes, ["annual", "kharif", "rabi", "zaid"]):
    fields = [f"{measure}{'' if period == 'annual' else '_' + period}_in_mm_{y}-{y+1}" for y in YEARS]
    record = annual if period == "annual" else seasonal
    series = pd.to_numeric(record.reindex(fields), errors="coerce")
    values.append(series)
    ax.plot(YEARS, series, marker="o")
    ax.set(title=f"{measure} · {period}", ylabel="mm", ylim=(0, None))
    ax.grid(alpha=0.2)
display(pd.concat(values).to_frame("value"))
axes[-1].set_xticks(YEARS, [f"{y}–{str(y+1)[-2:]}" for y in YEARS], rotation=45)
plt.tight_layout()
plt.show()


## One MWS time series

`get_mws_data` fetches only the selected MWS. Choose `water_field` and `ndvi_field` to explore other measures; the actual dates returned by the API are used for both.


In [ ]:
response = requests.get(API_URL + "get_mws_data/", params={**place, "mws_id": mws_id}, headers=api_headers, timeout=180)
series = pd.DataFrame(read_json(response)["time_series"])
series["date"] = pd.to_datetime(series["date"])
series = series.set_index("date").sort_index()
display(pd.DataFrame({"field": series.columns}))
water_field, ndvi_field = "precipitation", "ndvi_crop"
view = series.loc["2017-07-01":"2025-06-30", [water_field, ndvi_field]].apply(pd.to_numeric, errors="coerce")
display(view.head())
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
view[water_field].plot(ax=axes[0], title=water_field, ylabel="mm")
view[ndvi_field].plot(ax=axes[1], title=ndvi_field, ylabel="NDVI (unitless)", ylim=(-1, 1))
plt.tight_layout()
plt.show()


### Try another field

Set `water_field` to `et` or `runoff`, and `ndvi_field` to `ndvi_tree` or `ndvi_shrub`. To inspect groundwater using the earlier table example, choose `table_name = "soge_vector"` with `soge_dev_percent` and `class_name`, or `"aquifer_vector"` and inspect its columns. Annual `welldepth_in_m_2017-2018` and `deltag_in_mm_2017-2018` use different units; give them separate axes when plotting.
